loading the data and checking the first 5 row

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('P2/BOM.csv')

In [2]:
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


First, removing those samples in which the target attribute has unassigned value.

In [3]:
df = df.dropna(subset=['RainTomorrow'])

As for the remaining attributes with null values, you must set them to the median of the values of that
attribute across the same exact month. To accomplish this, you can define a new column
‘month’ which contains the corresponding month according to the value of ‘date’. Please note
that you can also use the values of this column as a feature. Finally, partition the dataset into
three sets ‘train’, ‘test’, and ‘validation’ according to appropriate proportions.

In [4]:
df['Date'] = pd.to_datetime(df['Date'])
df['month'] = df['Date'].dt.month

numeric_columns = df.select_dtypes(include=np.number).columns
for col in numeric_columns:
    # Loop through every month, from 1 to 12
    for month_num in range(1, 13):
        
        is_month = (df['month'] == month_num)        
        month_median = df.loc[is_month, col].median()
        df.loc[is_month, col] = df.loc[is_month, col].fillna(month_median)


df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

X = df.select_dtypes(include=np.number).drop(columns=['RainTomorrow'])
y = df['RainTomorrow']

partitioning the dataset into three sets ‘train’, ‘test’, and ‘validation’ 

In [5]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, random_state=42)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (99543, 17)
Val shape: (21321, 17)
Test shape: (21329, 17)


To prevent overfitting, we first aim to reduce the number of variables. One possible
method is to detect highly correlated features and eliminate either of them from the
dataset. We find the feature pairs, if any, with correlation coefficient over 0.95 and we remove them.

In [6]:
corr_matrix = X_train.corr().abs()

triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

cols_to_drop = [column for column in triangle.columns if any(triangle[column] > 0.95)]
print("Dropping features:", cols_to_drop)

X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)


Dropping features: ['Pressure3pm', 'Temp3pm']


Next, we determine the 10 best features of the dataset using sequential feature selection
method.

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SequentialFeatureSelector

knn_base = KNeighborsClassifier(n_neighbors=5)
sfs = SequentialFeatureSelector(knn_base, n_features_to_select=10)

sfs.fit(X_train, y_train)

best_features = list(X_train.columns[sfs.get_support()])
print("The 10 best features:", best_features)


X_train_10 = X_train[best_features]
X_val_10 = X_val[best_features]
X_test_10 = X_test[best_features]

The 10 best features: ['MinTemp', 'Rainfall', 'Sunshine', 'WindGustSpeed', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Cloud3pm', 'Temp9am']


Apply K–NN to the modified dataset. Find the best value of k using the validation set,
and report the accuracy of the model on the test set.

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_10)

X_val_scaled = scaler.transform(X_val_10)
X_test_scaled = scaler.transform(X_test_10)

best_k = 1
best_accuracy = 0

# K from 1 to 20
for k in range(1, 20):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    val_preds = knn.predict(X_val_scaled)
    acc = accuracy_score(y_val, val_preds)
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_k = k

print("Best K:", best_k)

final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_scaled, y_train)

test_preds = final_knn.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, test_preds)
print("Accuracy on Test Set:", round(test_accuracy, 4))

Best K: 19
Accuracy on Test Set: 0.8412


best k is 19.

In [9]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_10, y_train)

val_preds = knn.predict(X_val_10)
valacc = accuracy_score(y_val, val_preds)
testacc = accuracy_score(y_test, test_preds)
valacc, testacc

(0.8287134749777215, 0.8411552346570397)

In [10]:
knn = KNeighborsClassifier(n_neighbors=19)
knn.fit(X_train_10, y_train)

val_preds = knn.predict(X_val_10)
valacc = accuracy_score(y_val, val_preds)
testacc = accuracy_score(y_test, test_preds)
valacc, testacc

(0.8410018291824961, 0.8411552346570397)

Do you think is it possible to determine the probability of rainfall using the same strategy? If so, how?

Yes. Instead of using .predict() which outputs 0 or 1, we use the .predict_proba() function.
In K-NN, this calculates the probability based on the fraction of the 'K' nearest neighbors that belong to the 'Rain' class.